---
CS235 Methods Project Part 2
---


In [ ]:
#Download and unzip Dataset from kaggle as uci archive returns 502 bad gateway
#https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data
!curl -L -o breast-cancer-wisconsin-data.zip\
https://www.kaggle.com/api/v1/datasets/download/uciml/breast-cancer-wisconsin-data
!unzip breast-cancer-wisconsin-data.zip

Importing all the libraries and preprocessing the data for future as done in part 1

In [16]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer
from sklearn.preprocessing import StandardScaler

from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real

import plotly.express as px

# Global Variables for consistency
RANDOM_STATE = 1 
K_FOLD = 5

# Same as part 1

# Load the dataset
bcw_df = pd.read_csv('data.csv')
#Column Id and Unnamed: 32 are not useful, One is just Ids for reference and other appears to only contain NaNs
if 'Unnamed: 32' in bcw_df.columns and 'id' in bcw_df.columns:
    bcw_df.drop(columns=['id','Unnamed: 32'],inplace=True)
#Separate target and features before adding noise to the data
X = bcw_df.drop(columns=['diagnosis'])
y = bcw_df['diagnosis'].map(lambda value: 1 if value == 'M' else 0 )

# ---
# Stratified Train test split (80/20) for imbalanced dataset, to ensure that the even distribution of classes.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)


## 1. Hyperparameter optimization
For the Bayes search and grid search we keep few things constant like the random search and CV fold to have a consitent comparison!
For random we don't need standardize the data as it is..

In [17]:
OPTIMIZATION_ITERATIONS = 5
def get_best_optimization_results(model:GridSearchCV | BayesSearchCV):
    total_time = 0
    f1_scores = []
    top_params = None
    avg_time = 0
    for _ in range(OPTIMIZATION_ITERATIONS):
        start_time = time.time()
        model.fit(X_train,y_train)
        total_time += time.time() - start_time
        f1_scores.append(model.best_score_)
        top_params = model.best_params_
    avg_time = total_time/OPTIMIZATION_ITERATIONS
    return avg_time, np.mean(f1_scores),np.std(f1_scores), top_params

For bayesian and grid opt we define same grid space, event though for bayes a different type of search space is preffered. this is to keep conistency while doing comparison.
Parameters that we keep fix or use the default :
```max_leaf_nodes = None``` Redundant as we are already restricting via ```max_depth``` and ```min_samples_split```.
``` min_impurity_decrease = 0 (Default)``` Not needed as this is effective for large noisy dataset.
``` bootstrap=True(Default)``` Bootstrap sampling is preferred way for random forest.
``` oob_score = False(Default)``` We are using Stratified K fold.
```n_jobs = -1 ``` This will allow jobs to run in parallel across all the processors.
```random_state = 1``` Consistentcy across multiple runs
``` verbose = 0(Default)``` It will disable the log
``` warm_start = False(Default)``` We are fitting new models each iteration so not applicable.
``` min_weight_fraction_leaf = 0 (Default)``` Not needed as we are tunning ```min_samples_leaf```
``` class_weight = None (Default) ``` Not required for this dataset.
``` max_samples = None ``` As the data set is already small we can leave this.
``` ccp_alpha = 0.0``` Leaving it as default.


In [ ]:
# Parameter grids for both coarse and fine search

fine_param_grid = {
    'n_estimators': [50,75,100,125,150,200,250,300],
    # 'n_estimators': [100],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 4,6,8, 10],
    'min_samples_leaf': [1, 2,3, 4,5],
    'max_features': ['sqrt', 'log2', None],
}
coarse_param_grid = {
    'n_estimators': [100,200,300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 'log2', None],
}

bayes_fine_grid = {
    'n_estimators': Integer(100, 300),
    'max_depth': Categorical([None, 5, 10, 15, 20, 25]),
    'min_samples_split': Integer(2, 10),
    'min_samples_leaf': Integer(1, 5),
    'max_features': Categorical(['sqrt', 'log2', None]),
}
bayes_coarse_grid = {
    'n_estimators': Categorical([100, 200,300]),
    'max_depth': Categorical([None, 10, 20]),
    'min_samples_split': Categorical([5, 10]),
    'min_samples_leaf': Categorical([2, 4]),
    'max_features': Categorical(['sqrt', 'log2', None]),
}

#Define the scorer for grid search as F1
scorer = make_scorer(f1_score)
cv = StratifiedKFold(n_splits=K_FOLD, shuffle=True, random_state=RANDOM_STATE)

random_forest_model = RandomForestClassifier(random_state=RANDOM_STATE)
grid_search_fine = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), fine_param_grid, scoring=scorer, cv=cv,n_jobs=-1)
average_time_grid_search_fine, mean_f1_score_grid_search_fine, f1_std_grid_search_fine, top_params_grid_search_fine = get_best_optimization_results(grid_search_fine)

grid_search_coarse = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), coarse_param_grid, scoring=scorer, cv=cv,n_jobs=-1)
average_time_grid_search_coarse, mean_f1_score_grid_search_coarse, f1_std_grid_search_coarse, top_params_grid_search_coarse = get_best_optimization_results(grid_search_coarse)

bayes_opt_fine = BayesSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), bayes_fine_grid, scoring=scorer, cv=cv, n_iter=50, random_state=RANDOM_STATE,n_jobs=-1)
average_time_bayes_opt_fine, mean_f1_score_bayes_opt_fine, f1_std_bayes_opt_fine, top_params_bayes_opt_fine = get_best_optimization_results(bayes_opt_fine)

bayes_opt_coarse = BayesSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), bayes_coarse_grid, scoring=scorer, cv=cv, n_iter=50, random_state=RANDOM_STATE,n_jobs=-1)
average_time_bayes_opt_coarse, mean_f1_score_bayes_opt_coarse, f1_std_bayes_opt_coarse, top_params_bayes_opt_coarse = get_best_optimization_results(bayes_opt_coarse)

print('---')
print(f'Grid Search (Fine)    | Avg Time: {average_time_grid_search_fine:.2f}s | F1 Score: {mean_f1_score_grid_search_fine:.4f} +/- {f1_std_grid_search_fine:.4f} | Top Params: {top_params_grid_search_fine}')
print(f'Grid Search (Coarse)  | Avg Time: {average_time_grid_search_coarse:.2f}s | F1 Score: {mean_f1_score_grid_search_coarse:.4f} +/- {f1_std_grid_search_coarse:.4f} | Top Params: {top_params_grid_search_coarse}')
print(f'Bayes Search (Fine)   | Avg Time: {average_time_bayes_opt_fine:.2f}s | F1 Score: {mean_f1_score_bayes_opt_fine:.4f} +/- {f1_std_bayes_opt_fine:.4f} | Top Params: {top_params_bayes_opt_fine}')
print(f'Bayes Search (Coarse) | Avg Time: {average_time_bayes_opt_coarse:.2f}s | F1 Score: {mean_f1_score_bayes_opt_coarse:.4f} +/- {f1_std_bayes_opt_coarse:.4f} | Top Params: {top_params_bayes_opt_coarse}')
print('---')

---
Grid Search (Fine)    | Avg Time: 257.00s | F1 Score: 0.9560 +/- 0.0000 | Top Params: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 75}
Grid Search (Coarse)  | Avg Time: 12.24s | F1 Score: 0.9503 +/- 0.0000 | Top Params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 300}
Bayes Search (Fine)   | Avg Time: 26.03s | F1 Score: 0.9588 +/- 0.0000 | Top Params: OrderedDict({'max_depth': 25, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 103})
Bayes Search (Coarse) | Avg Time: 24.23s | F1 Score: 0.9503 +/- 0.0000 | Top Params: OrderedDict({'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 300})
---


## 2. Data Augmentation